<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
Capstone
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** Der Capstone ist kein freies KI-Projekt, sondern der Nachweis, dass ein eigenes System als **kontrolliertes Agentensystem** funktioniert — mit demselben Dreiklang wie der Kurs: **Planen** (Tool-/Worker-Wahl und State), **Handeln** (Tools, RAG und Workflow) und **Pruefen** (Gate/HITL, Evaluation mit Negativfaellen). Die Pflichtbestandteile unten sind bewusst an den Capstone-Kriterien der Kurs-Leitaufgabe ausgerichtet.

In [1]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul
!uv pip install --system -q fastapi uvicorn httpx

import os
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "M36-Capstone"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)

setup_api_keys(['OPENAI_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS
# LangSmith Tracing
run_cfg = {
    "run_name": "M36_Capstone",
    "tags": ["m36", "capstone"],
    "metadata": {"notebook": "M36", "version": "1.0"}
}


✓ OPENAI_API_KEY erfolgreich gesetzt

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.15
langchain-chroma                         1.1.0
langchain-classic                        1.0.4
langchain-community                      0.4.1
langchain-core                           1.3.1
langchain-ollama                         1.1.0
langchain-openai                         1.2.0
langchain-text-splitters                 1.1.2
langgraph                                1.1.9
langgraph-checkpoint                     4.0.2
langgraph-prebuilt                       1.0.10
langgraph-sdk                            0.3.13

IP-Adresse: 34.46.3.32
Hostname: 32.3.46.34.bc.googleusercontent.com
Stadt: Council Bluffs
Region: Iowa
Land: US
Koordinaten: 41.2619,-95.8608
Provider: AS396982 Google LLC
Postleitzahl: 51502
Zeitzone: America/Chicago


# 1 | Kursrückblick: Agenten-Architektur

---




Dieses Diagramm strukturiert das Konzept eines Agenten entlang von drei Ebenen: **Eigenschaften**, **Funktionen** und **Infrastruktur**.

Auf der ersten Ebene werden grundlegende **Merkmale** beschrieben, die einen Agenten auszeichnen, etwa Autonomie, Reaktionsfähigkeit oder Zielorientierung.

Daraus leiten sich auf der zweiten Ebene die zentralen **Fähigkeiten** ab, die ein Agent zur Aufgabenerfüllung benötigt – beispielsweise Planung, Nutzung von Werkzeugen oder der Umgang mit Zustand und Feedback.

Die dritte Ebene umfasst schließlich die **technischen** **Rahmenbedingungen**, die einen stabilen und kontrollierten Betrieb ermöglichen, wie Kontextverwaltung, Zugriff auf externe Systeme oder Sicherheitsmechanismen.

Die Zuordnung der Module (Mx) dient dabei als Referenz auf die zugrunde liegenden Inhalte und erlaubt eine gezielte Vertiefung einzelner Aspekte.



In [2]:
#@markdown   <p><font size="4" color='green'>  Agenten-Architektur: Eigenschaften, Funktionen & Infrastruktur</font> </br></p>

diagram = """
%%{init: {'theme':'forest'}}%%
graph TD
    classDef property fill:#ececff,stroke:#9370db,stroke-width:2px,color:#333
    classDef function fill:#e1f5fe,stroke:#01579b,stroke-width:2px,color:#333
    classDef infra fill:#fff3e0,stroke:#e65100,stroke-width:2px,color:#333
    classDef groupStyle fill:#f9f9f9,stroke:#d3d3d3,stroke-dasharray: 5 5

    subgraph E1["<b>🧠 1. Eigenschaften (Was einen Agenten ausmacht)</b>"]
        direction LR
        A(["<b>Autonomie</b><br/>Trifft eigenständig Entscheidungen<br/><sub>M03, M08</sub>"])
        R(["<b>Reaktionsfähigkeit</b><br/>Reagiert auf Änderungen<br/><sub>M02, M06, M10</sub>"])
        P(["<b>Proaktivität</b><br/>Verfolgt Ziele über Zeit<br/><sub>M19–M21</sub>"])
        I(["<b>Interaktionsfähigkeit</b><br/>Interagiert mit Menschen und Systemen<br/><sub>M04, M17</sub>"])
    end

    subgraph E2["<b>🛠️ 2. Funktionen (Was ein Agent können muss)</b>"]
        direction LR
        PL["<b>Planning</b><br/>Zerlegt Ziele in Schritte<br/><sub>M09, M32</sub>"]
        ME["<b>Memory</b><br/>Hält Zustand und Verlauf<br/><sub>M16, M18</sub>"]
        TU["<b>Tool Use</b><br/>Nutzt externe Werkzeuge und APIs<br/><sub>M02, M06, M30</sub>"]
        MO["<b>Monitoring</b><br/>Beobachtet Ergebnisse und Feedback<br/><sub>M15, M24</sub>"]
        DE{{"<b>Delegation (optional)</b><br/>Orchestriert Sub-Agenten<br/><sub>M20, M32, M33, M36</sub>"}}
    end

    subgraph E3["<b>🏗️ 3. Infrastruktur (Was den Betrieb stützt)</b>"]
        direction LR
        CM[/"<b>Context Management</b><br/>Filtert und verdichtet Kontext<br/><sub>M11–M14, M22</sub>"/]
        GR[/"<b>Guardrails</b><br/>Prüft und begrenzt Aktionen<br/><sub>M10, M23</sub>"/]
        EA[/"<b>Environment Access</b><br/>Dateien, APIs, Systeme<br/><sub>M28, M30, M35</sub>"/]
        OB[/"<b>Observability</b><br/>Logs, Traces, Debugging<br/><sub>M15, M24, M35, M36</sub>"/]
    end

    A -.-> PL
    A -.-> TU

    P -.-> PL
    P -.-> ME
    P -.-> DE

    R -.-> MO

    I -.-> TU

    PL --> CM
    ME --> CM

    TU --> EA
    TU --> GR

    MO --> OB
    DE --> OB

    class A,R,P,I property
    class PL,ME,TU,MO,DE function
    class CM,GR,EA,OB infra
    class E1,E2,E3 groupStyle
"""

mermaid(diagram, width=1300)

# 2 | Agent-Pattern
---

Dieser Kurs hat sich Schritt für Schritt durch das Agenten-Ökosystem geführt.
Viele Konzepte haben **Namen** — Pattern-Bezeichnungen aus der AI-Engineering-Literatur.

Das folgende Diagramm zeigt, welche Kursinhalte welchen etablierten Patterns entsprechen.

<p><font color='darkblue' size="4">📝 <b>Hinweis</b></font></p>

Pattern-Namen sind Vokabular — für Gespräche, Interviews, Architektur-Diskussionen. Das Verständnis entsteht durch die praktische Arbeit in den Modulen — die Namen kommen danach.

In [3]:
#@markdown   <p><font size="4" color="green">Pattern-Landkarte nach Gruppen</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
mindmap
  root((Generative AI
Design Patterns))
    Agent-Grundlagen
      Tool Calling
      Code Execution
      Structured Output
      Dependency Injection
      Prompt Optimization
    Reasoning
      Chain of Thought
    Output Control
      Style Transfer
      Template Generation
    Memory und Kontrolle
      Persistentes Memory
      user-in-the-Loop
    RAG und Wissen
      Basic RAG
      Semantic Indexing
      Indexing at Scale
      Index-aware Retrieval
      Node Postprocessing
      Trustworthy Generation
      Deep Search
    Zuverlaessigkeit
      Reflection
      LLM-as-Judge
    Multi-Agent
      Multi-Agent Collaboration
    Deployment
      Small Language Model
      Prompt Caching
      Inference Optimization
      Degradation Testing
    Safety
      Self-Check
      Guardrails
'''

mermaid(diagram, width=1000)


## Pattern-Übersicht

> Quelle: Lakshmanan & Hapke — *Generative AI Design Patterns* (O'Reilly)
> **—** = im Kurs nicht behandelt &nbsp;|&nbsp; **\*** = kursinternes Pattern (nicht im Buch)


### Agent-Grundlagen

| Pattern | Kurz-Erläuterung | Modul |
|---|---|---|
| **Tool Calling** | LLM ruft externe Funktionen und APIs auf | M02, M03, M06 |
| **Code Execution** | Agent führt Code aus und interpretiert Ergebnisse | — |
| **Structured Output** | Typsichere Ausgaben via Pydantic / JSON-Schema | M05 |
| **Dependency Injection** | Kontext und Daten dynamisch in Prompts einbetten | M04 |
| **Prompt Optimization** | Systematische Verbesserung von System-Prompts | M04 |



### Reasoning

| Pattern | Kurz-Erläuterung | Modul |
|---|---|---|
| **Chain of Thought** | Schritt-für-Schritt-Reasoning im Prompt | M04 |



### Output Control

| Pattern | Kurz-Erläuterung | Modul |
|---|---|---|
| **Style Transfer** | Textstil via Prompt gezielt verändern | M04 |

| **Template Generation** | Strukturierte Ausgabe-Templates erzwingen | M04, M05 |



### Memory & Kontrolle

| Pattern | Kurz-Erläuterung | Modul |
|---|---|---|
| **Persistentes Memory** | Persistentes Gedächtnis über Sitzungen hinweg | M16, M18 |
| **user-in-the-Loop** \* | Mensch genehmigt oder korrigiert Agenten-Aktionen | M17 |


### RAG & Wissen

| Pattern | Kurz-Erläuterung | Modul |
|---|---|---|
| **Basic RAG** | Dokumenten-Retrieval + LLM-Antwortgenerierung | M11, M12, M13 |
| **Semantic Indexing** | Vektorbasierte Indexierung für semantische Suche | M11, M12 |
| **Indexing at Scale** | Skalierbare Indizierung großer Dokumentenmengen | M12 |
| **Index-aware Retrieval** | Retrieval mit Kenntnis der Index-Struktur | M13, M14 |
| **Node Postprocessing** | Filtern und Reranken abgerufener Dokumente | M27 |
| **Trustworthy Generation** | Belegbare Antworten mit Quellen-Grounding | M27 |
| **Deep Search** | Mehrschrittige, adaptive Suche mit Agenten | M14, M22, M27 |


### Zuverlässigkeit & Evaluation

| Pattern | Kurz-Erläuterung | Modul |
|---|---|---|
| **Reflection** | Agent prüft und korrigiert eigene Ausgaben | M17, M27 |
| **LLM-as-Judge** | LLM bewertet automatisiert Qualität von Ausgaben | M15, M24 |


### Multi-Agent

| Pattern | Kurz-Erläuterung | Modul |
|---|---|---|
| **Multi-Agent Collaboration** | Spezialisierte Agenten arbeiten koordiniert | M19, M20, M21, M34 |


### Deployment

| Pattern | Kurz-Erläuterung | Modul |
|---|---|---|
| **Small Language Model** | Kleinere Modelle für spezifische Teilaufgaben | M35 |
| **Prompt Caching** | Wiederverwendung gecachter Prompt-Präfixe | M35 |
| **Inference Optimization** | Batching, Quantisierung, Latenzoptimierung | M35 |
| **Degradation Testing** | Belastungs- und Regressionstests im Deployment | M35 |


### Safety

| Pattern | Kurz-Erläuterung | Modul |
|---|---|---|
| **Self-Check** | Agent validiert eigene Ausgaben vor der Rückgabe | M23 |
| **Guardrails** | Sicherheitsschranken für Input und Output | M23 |


# A | Aufgabe

---


<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Das Capstone ist die Abschlussarbeit des Kurses. Es gibt kein vorgegebenes Lösungsmodell — nur Mindestanforderungen und optionale Erweiterungen.

**Hinweis zur Lösungshilfe:**
> Gemini in Google Colab, LangSmith-Tracing und die Notebooks aus dem Kurs stehen als Unterstützung zur Verfügung.

---

## Pflichtbestandteile

Ausgerichtet an den Capstone-Kriterien der Kurs-Leitaufgabe: Das Projekt wird nicht als allgemeine KI-Anwendung bewertet, sondern als kontrolliertes Agentensystem.

| # | Anforderung | Kriterium |
|---|-------------|-----------|
| 1 | **Architektur-Skizze** (Mermaid) | Alle Komponenten beschriftet, Pattern-Namen angegeben |
| 2 | **Kontrollierter Workflow** | Supervisor + ≥ 2 Worker als StateGraph, insgesamt ≥ 3 Tools (davon mind. 1 RAG/Evidence Tool) |
| 3 | **Strukturierte Ausgabe** | Finale Antwort als Pydantic-Schema (z. B. Antwort, Quelle, Sicherheit) |
| 4 | **Freigabe-/Kontrollpunkt** | Mindestens ein Gate oder HITL-Schritt vor einer kritischen Ausgabe |
| 5 | **Evaluation mit Negativfällen** | ≥ 1 LLM-as-Judge-Kriterium **und** mindestens ein Testfall, bei dem das System korrekt ablehnt oder eskaliert |
| 6 | **Sichtbare Quellen/Traces** | Tool-Aufrufe und Quellen im LangSmith-Trace oder in der Ausgabe nachvollziehbar |

## Optionale Erweiterungen

- Mehrere RAG-Quellen oder Vektordatenbank-Typen
- Gradio UI für den Agenten (Quellen, Trace, Freigabe sichtbar)
- Fehlertoleranz (Retry-Logik, Fallback-Agenten)
- LangSmith Evaluation Dataset mit Baseline-Score

## Bewertungskriterien

| Kriterium | Punkte |
|-----------|--------|
| Architektur sinnvoll & begründet | 2 |
| Pattern-Namen korrekt verwendet | 1 |
| Code lauffähig & strukturiert | 2 |
| Strukturierte Ausgabe vorhanden | 1 |
| Gate/HITL-Schritt vorhanden | 2 |
| Evaluation inkl. Negativfall implementiert | 2 |
| **Gesamt** | **10** |

## Abgabeformat

- Lauffähiges Jupyter Notebook (alle Zellen ausgeführt)
- Mermaid-Diagramm als erste Zelle im Implementierungsblock
- Kurze Markdown-Zelle am Ende mit zwei Reflexionsfragen: Was würdest du als nächstes verbessern? Wann darf der Agent nicht autonom handeln?

**✅ Erledigt wenn:** Das Notebook läuft von oben nach unten fehlerfrei durch; Mermaid-Diagramm, Pattern-Tabelle, strukturierte Ausgabe, Gate/HITL-Schritt und mindestens ein LLM-as-Judge-Ergebnis mit Negativfall sind vorhanden.


## Starter-Setup: Korpus und Eval-Set

Nutzen Sie diesen Block als Ausgangspunkt für den Research-Assistant-Capstone.


In [ ]:
from pathlib import Path
import json
import shutil
from genai_lib.utilities import copy_from_github

KORPUS_QUELLE = "ralf-42/Agenten/02_daten/01_text/korpus_research"
KORPUS_MASKE = "*.pdf"
KORPUS_TARGET = "/content/files"

EVAL_QUELLE = "ralf-42/Agenten/02_daten/05_sonstiges"
EVAL_MASKE = "eval_research*.json"
EVAL_TARGET = "/content/eval"

shutil.rmtree(KORPUS_TARGET, ignore_errors=True)
copy_from_github(
    source=KORPUS_QUELLE,
    target=KORPUS_TARGET,
    mask=KORPUS_MASKE,
)

shutil.rmtree(EVAL_TARGET, ignore_errors=True)
copy_from_github(
    source=EVAL_QUELLE,
    target=EVAL_TARGET,
    mask=EVAL_MASKE,
)

with open(Path(EVAL_TARGET) / "eval_research.json", encoding="utf-8") as f:
    eval_research = json.load(f)

print(f"Korpus geladen nach: {KORPUS_TARGET}")
print(f"Eval-Fragen geladen: {len(eval_research)}")


In [ ]:
# Capstone-Implementierung
# Tipp: Mermaid-Diagramm als Blaupause verwenden

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

# 1. Modelle initialisieren
# supervisor_llm = init_chat_model(JUDGE)   # gpt-5.4 empfohlen
# worker_llm     = init_chat_model(WORKER)  # gpt-5.4-mini, kein temperature
# judge_llm      = init_chat_model(JUDGE)   # gpt-5.4 empfohlen

# 2. Tools definieren
# at.tool
# def mein_tool(eingabe: str) -> str: ...

# 3. Agenten aufbauen (Supervisor + mindestens 2 Worker)

# 4. StateGraph zusammensetzen

# 5. LangSmith-Tracing konfigurieren
# import os
# os.environ['LANGSMITH_TRACING'] = 'true'
# os.environ['LANGSMITH_PROJECT'] = 'capstone'

# 6. LLM-as-Judge für kritische Stelle

# 7. System testen
# result = mein_system.invoke({'input': 'Testaufgabe'}, config=run_cfg)
# print(result)

# --- Was würde ich als nächstes verbessern? ---
# ausblick = '...'